## Step 1: Bronze Layer — GCS CSV → Delta Lake

**Bronze 层的职责**: 从 GCS 读取原始 CSV，无损贴源写入 Delta Lake。

**设计原则**:
- 只做格式转换（CSV → Delta），不做任何清洗、去重、Join
- 使用 `inferSchema` 自动推断列类型，不手动 CAST（故意保留原始数据面貌）
- 以 Delta 格式存储，获得 ACID 事务 + 时间旅行 + Schema 演化能力
- 清洗和标准化留给 Silver 层

**数据源**: `gs://yuto-olist_raw_data/`（9 张 Olist CSV，平铺在 bucket 根目录）

**输出**: 9 张 Delta 表，存入 Databricks hive_metastore 的 `default` schema

In [0]:
# ============================================================
# 配置区：定义 GCS 数据源路径和 9 张表的映射关系
# ============================================================

# GCS bucket 名称——你的 Olist 原始 CSV 全部平铺在这个 bucket 根目录下
GCS_BUCKET = "gs://yuto-olist_raw_data"

# CSV 文件名 → Bronze Delta 表名的映射字典
# 命名规范：所有 Bronze 表统一加 bronze_ 前缀，便于后续 Silver 层引用
# 注意：这里只做格式转换，列名和列类型完全保留原始状态
TABLE_MAPPING = {
    "olist_orders_dataset.csv":              "bronze_orders",
    "olist_customers_dataset.csv":           "bronze_customers",
    "olist_order_items_dataset.csv":         "bronze_order_items",
    "olist_products_dataset.csv":            "bronze_products",
    "olist_sellers_dataset.csv":             "bronze_sellers",
    "olist_order_payments_dataset.csv":      "bronze_payments",
    "olist_order_reviews_dataset.csv":       "bronze_reviews",
    "olist_geolocation_dataset.csv":         "bronze_geolocation",
    "product_category_name_translation.csv": "bronze_translation",
}

print(f"📂 GCS Bucket: {GCS_BUCKET}")
print(f"📋 待导入表数: {len(TABLE_MAPPING)}")

In [0]:
# ============================================================
# 核心函数：单张 CSV → Delta Bronze 表
# ============================================================

def ingest_csv_to_bronze(file_name, table_name):
    """
    从 GCS 读取单个 CSV 文件，写入 Databricks 的 Delta Lake Bronze 表。

    参数:
        file_name:  GCS bucket 下的 CSV 文件名（如 "olist_orders_dataset.csv"）
        table_name: 目标 Delta 表名（如 "bronze_orders"）

    设计决策:
        - header="true": CSV 第一行是列名，不是数据
        - inferSchema="true": 让 Spark 自动推断列类型（Bronze 层不做手动 CAST）
        - mode("overwrite"): 全量覆盖——Olist 是静态快照，没有增量追加需求
        - 写入 Delta 格式而非 Parquet，因为 Delta 额外提供：
          ① ACID 事务保证（不会写一半失败留下脏数据）
          ② 时间旅行（可以回看历史版本）
          ③ Schema 演化（后续加列不会报错）
    """
    # 拼接完整的 GCS 路径
    gcs_path = f"{GCS_BUCKET}/{file_name}"
    print(f"📥 正在读取: {gcs_path}")

    # Spark CSV Reader：从 GCS 读取 CSV，自动推断 Schema，不做任何清洗
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(gcs_path)

    # 写入 Delta Lake 表
    # mode("overwrite") 每次完全覆盖——Olist 数据是静态快照，不需要增量
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)

    # 打印行数确认导入成功
    row_count = df.count()
    print(f"✅ {table_name} 写入完成 — {row_count:,} 行")

In [0]:
# ============================================================
# 批量执行：遍历 9 张表，依次从 GCS 读入并写入 Bronze 层
# ============================================================

# 使用 for 循环批量调用上面的函数，原因：
# - 比逐个手写调用更易维护（新增/删除表只需改 TABLE_MAPPING 字典）
# - 每个文件的失败不会中断其他文件的导入（如果需要更强的容错，可以加 try/except）
for file_name, table_name in TABLE_MAPPING.items():
    ingest_csv_to_bronze(file_name, table_name)

print("\n🎉 全部 9 张 Bronze 表写入完成！")

In [0]:
# ============================================================
# 验证：列出所有新创建的 Bronze 表，确认数量和名称
# ============================================================

# 查询 hive_metastore 的 default schema 下所有 bronze_ 开头的表
# 预期输出：9 张表，从 bronze_orders 到 bronze_translation
print("📋 Bronze 层已注册的表：")
spark.sql("SHOW TABLES IN default LIKE 'bronze_*'").show(20, False)

---

### 运行前提：Databricks 需要能访问你的 GCS Bucket

如果运行报 `com.google.cloud.spark.bigquery.repackaged.com.google.api.gax...` 等认证错误，
说明 Databricks 集群还没有配置 GCS 访问权限。两种解决方式：

1. **推荐**: 在 Databricks 集群配置中绑定 GCP Service Account（`Compute → Advanced Options → Google Service Account`）
2. **临时方案**: 在 notebook 中手动设置认证：
   ```python
   spark.conf.set("fs.gs.auth.service.account.email", "your-sa@project.iam.gserviceaccount.com")
   spark.conf.set("fs.gs.auth.service.account.keyfile", "/path/to/key.json")
   ```

### 下一步

Bronze 表创建完成后，进入 [02_silver/]() 做数据清洗和 JOIN。